# Chapter 74: Advanced Experiment Design (Uplift + Bandits Hybrid)

## Learning Objectives
- Combine uplift targeting with online exploration
- Compare A/B, uplift-only, and hybrid policies
- Measure cumulative reward by policy mix
- Analyze segment-level performance

## Prerequisites
- Chapters 0-70 completed
- Strong practical understanding of causal inference, ranking, and MLOps
- Optional matplotlib for richer plots


# Chapter 74: Advanced Experiment Design (Uplift + Bandits Hybrid)

## Why this chapter matters
Many real systems need both causal uplift targeting and online exploration. Hybrid policies can improve gains while continuously learning.

## Learning goals
1. Build uplift score based targeting policy.
2. Blend uplift targeting with contextual Thompson sampling.
3. Compare fixed A/B, uplift-only, and hybrid bandit policies.
4. Evaluate cumulative gain and regret.

## Hybrid policy intuition
1. estimate uplift propensity by segment/user features
2. prioritize high-uplift candidates
3. within candidate set, use bandit sampling for exploration

## Assignment
1. Sweep hybrid mixing coefficient.
2. Evaluate stability under non-stationary response.
3. Report policy performance by user segment.


In [ ]:
from __future__ import annotations

import random
from typing import Dict, List, Tuple


def sample_segment() -> int:
    r = random.random()
    if r < 0.35:
        return 0
    if r < 0.7:
        return 1
    return 2


def uplift_true(seg: int) -> float:
    return {0: 0.03, 1: 0.08, 2: -0.01}[seg]


def base_response(seg: int) -> float:
    return {0: 0.09, 1: 0.11, 2: 0.07}[seg]


def reward(seg: int, treat: int) -> int:
    p = base_response(seg) + (uplift_true(seg) if treat == 1 else 0.0)
    p = max(0.001, min(0.999, p))
    return 1 if random.random() < p else 0


def run_ab_test(T=12000, treat_prob=0.5, seed=42):
    random.seed(seed)
    total = 0
    for _ in range(T):
        seg = sample_segment()
        tr = 1 if random.random() < treat_prob else 0
        total += reward(seg, tr)
    return total


def run_uplift_policy(T=12000, seed=42):
    random.seed(seed)
    total = 0
    for _ in range(T):
        seg = sample_segment()
        # Treat only if uplift expected positive
        tr = 1 if uplift_true(seg) > 0 else 0
        total += reward(seg, tr)
    return total


def run_hybrid_bandit(T=12000, mix=0.7, seed=42):
    random.seed(seed)

    # Thompson posteriors per segment for treatment arm effect (binary reward modeling)
    alpha_t = {0: 1.0, 1: 1.0, 2: 1.0}
    beta_t = {0: 1.0, 1: 1.0, 2: 1.0}
    alpha_c = {0: 1.0, 1: 1.0, 2: 1.0}
    beta_c = {0: 1.0, 1: 1.0, 2: 1.0}

    total = 0
    cumulative = []

    for _ in range(T):
        seg = sample_segment()

        # Uplift prior signal (could be model estimate in real system)
        uplift_est = uplift_true(seg)
        uplift_action = 1 if uplift_est > 0 else 0

        # Bandit action via Thompson on segment-specific treatment vs control
        p_t = random.betavariate(alpha_t[seg], beta_t[seg])
        p_c = random.betavariate(alpha_c[seg], beta_c[seg])
        bandit_action = 1 if p_t >= p_c else 0

        # Hybrid mix
        if random.random() < mix:
            action = uplift_action
        else:
            action = bandit_action

        r = reward(seg, action)
        total += r

        if action == 1:
            alpha_t[seg] += r
            beta_t[seg] += 1 - r
        else:
            alpha_c[seg] += r
            beta_c[seg] += 1 - r

        cumulative.append(total)

    return total, cumulative


def maybe_plot(mixes: List[float], vals: List[float]):
    try:
        import matplotlib.pyplot as plt
    except Exception:
        print("Hybrid mix performance:", list(zip([round(m, 2) for m in mixes], [round(v, 4) for v in vals])))
        return

    plt.figure(figsize=(7, 4))
    plt.plot(mixes, vals, marker="o")
    plt.xlabel("Uplift-policy mix weight")
    plt.ylabel("Average reward per round")
    plt.title("Hybrid Uplift+Bandit Policy")
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    T = 14000

    ab = run_ab_test(T=T, treat_prob=0.5, seed=9)
    up = run_uplift_policy(T=T, seed=9)

    mixes = [0.0, 0.25, 0.5, 0.75, 1.0]
    per_round = []
    for m in mixes:
        total, _ = run_hybrid_bandit(T=T, mix=m, seed=9)
        per_round.append(total / T)
        print(f"Hybrid mix={m:.2f} total={total} avg={total/T:.5f}")

    print("A/B baseline total:", ab, "avg:", round(ab / T, 5))
    print("Uplift-only total :", up, "avg:", round(up / T, 5))

    maybe_plot(mixes, per_round)


## Checkpoint

1. You can design hybrid exploration-exploitation policy.
2. You can compare policy outcomes under heterogeneous uplift.
3. You can tune mix coefficient for performance.

## Guided Exercise
Introduce non-stationarity and re-evaluate best hybrid mix.

In [ ]:
# TODO (Student Exercise)
# Modify uplift_true over time and compare policy robustness.

## Quick Quiz

**Q1.** Why combine uplift and bandits?

**Answer:** Uplift gives targeting prior, bandits maintain online adaptation.

**Q2.** What does mix weight control?

**Answer:** Balance between deterministic uplift policy and exploratory bandit policy.

**Q3.** Why evaluate by segment?

**Answer:** Policy gains and harms can vary strongly by user segment.


## Homework
Add regret tracking against oracle segment policy.